In [94]:
import requests
import json
from pathlib import Path
from datetime import datetime

In [95]:
CLIENT_ID = "F6ANuI3MVaRYJTwDd5IBF"
CLIENT_SECRET = "ufUDDt54UlVhK49vH3zl2ZM9lkG67aAfqpXpZxEZ"

DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [96]:
url = f"https://data.api.xweather.com/tropicalcyclones?p=&filter=all&limit=10&client_id={CLIENT_ID}&client_secret={CLIENT_SECRET}"
response = requests.get(url)

In [97]:
if response.status_code == 200:
    data = response.json()

    file_name = DATA_DIR / f"xweather_storms_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(file_name, "w") as f:
        json.dump(data, f, indent=2)
    print(f"[OK] Saved to {file_name}")
else:
    print(f"[ERROR] {response.status_code}: {response.text}")
    data = None

[OK] Saved to ..\data\raw\xweather_storms_20250907_183424.json


In [98]:
import pandas as pd

In [99]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [100]:
files = sorted(RAW_DIR.glob('xweather_storms_*.json'))
if not files:
    raise FileNotFoundError
lastest_file = files[-1]

with open(lastest_file, 'r') as f:
    data = json.load(f)


In [131]:
def extract_storm(data):
    storms = data.get('response', [])
    all_records = []

    for storm in storms:
        info = storm.get('profile', {})
        loc =  storm.get('position', {}).get('location', {}).get("coordinates", [None, None])

        all_records.append({
        'storm_id': storm.get("id"),
        'name': info.get('name'),
        'start_time': info.get("lifespan", {}).get("startDateTimeISO"),
        'basin': info.get('basinCurrent'),
        'event': info.get('event'),
        'storm_type': info.get('maxStormType'),
        'storm_cat': info.get('maxStormCat'),
        'lon': loc[0],
        'lat': loc[1]
    })
        
    return all_records

storm_record = extract_storm(data)

df_storm = pd.DataFrame(storm_record)
df_storm.head() 

df_storm.to_csv('../data/processed/storm.csv', index=False, header=True)

In [124]:
def extract_all_tracks(data):
    storms = data.get("response", [])
    all_records = []
    
    for storm in storms:  
        storm_id = storm.get("id")
        tracks = storm.get("track", [])
        for track in tracks:
            details = track.get("details", {})
            coords = track.get("location", {}).get("coordinates", [None, None])
            all_records.append({
                "storm_id": storm_id,
                "track_name": details.get("stormName"),
                "storm_type": details.get("stormType"),
                "storm_cat": details.get("stormCat"),
                "advisory": details.get("advisoryNumber"),
                'directionDEG': details.get('movement', {}).get('directionDEG'),
                'speed': details.get('movement', {}).get('speedKTS'),
                'wind_speed': details.get('windSpeedKPH'),
                'gust_speed': details.get('gustSpeedKPH'),
                'pressure': details.get('pressureMB'),
                "lon": coords[0],
                "lat": coords[1],
            })
    return all_records

track_records = extract_all_tracks(data)

df_tracks = pd.DataFrame(track_records)
df_tracks.head(10)
df_tracks.to_csv('../data/processed/track.csv', index=False, header=True)


In [133]:
import psycopg2

conn = psycopg2.connect("dbname=xweather_db user=postgres password=postgres host=localhost port=5432")
cur = conn.cursor()

with open("../data/processed/storm.csv", "r", encoding="utf-8") as f:
    cur.copy_expert("COPY storm FROM STDIN WITH CSV HEADER", f)

with open("../data/processed/track.csv", "r", encoding="utf-8") as f:
    cur.copy_expert("COPY track (storm_id, track_name, storm_type, storm_cat, advisory, directionDEG, speed, wind_speed, gust_speed, pressure, lon, lat) FROM STDIN WITH CSV HEADER", f)

conn.commit()
cur.close()
conn.close()

